# NB-10-02 — Tabular Control: Q-Learning and SARSA

**Module M10 · Reinforcement Learning** · [L10-03](../lessons/L10-03-temporal-difference-learning.md)

Learn from experience without a model, then use a task built to expose the difference between on-policy and
off-policy control. Cliff walking is that task: the optimal path is one step from a −100 penalty.

In [1]:
import sys
sys.path.insert(0, "../code")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rl_envs, dp, tabular, experiments

print("experiments self-test:", experiments.run_tests())

experiments self-test: True


## 1. Q-learning on a small grid

Four hundred episodes, fixed seed. The tail mean is the number to report; the first episodes are noise.

In [2]:
env = experiments.make_grid(4)
run = tabular.q_learning(env, episodes=400, alpha=0.5, gamma=0.9, eps=0.2, seed=0)
print("tail return:", round(experiments.mean_tail(run["returns"]), 3))
print("first 10:", np.round(run["returns"][:10], 2))
print("random baseline:", round(experiments.random_baseline(env, episodes=30, seed=0)["mean"], 3))
print("greedy path:", tabular.greedy_path_from_q(env, run["Q"], start=0))

tail return: 0.747
first 10: [-0.64  0.52  0.68  0.32 -0.04  0.32  0.6   0.8   0.68  0.8 ]
random baseline: -1.409
greedy path: [0, 1, 5, 9, 10, 11, 15]


## 2. Exploration matters

With $\varepsilon = 0$ the agent never discovers the goal. This is the cleanest demonstration that
exploration is part of the algorithm, not a detail.

In [3]:
for eps in (0.0, 0.05, 0.2, 0.5):
    r = tabular.q_learning(env, episodes=300, alpha=0.5, gamma=0.9, eps=eps, seed=0)
    print(f"eps={eps}: tail {experiments.mean_tail(r['returns']):6.2f}")

eps=0.0: tail   0.80
eps=0.05: tail   0.78


eps=0.2: tail   0.75


eps=0.5: tail   0.59


## 3. Cliff walking: the task that separates the two algorithms

−1 per step, −100 and episode end in the pit, goal at bottom-right. SARSA values the exploring policy;
Q-learning values the greedy one.

In [4]:
result = experiments.compare_sarsa_qlearning(episodes=300, seeds=(0, 1, 2))
print("SARSA tail:      ", round(result["sarsa_tail"], 2))
print("Q-learning tail: ", round(result["q_learning_tail"], 2))
print("per seed:", {k: [round(v, 1) for v in vals] for k, vals in result["tail_per_seed"].items()})

SARSA tail:       -32.42
Q-learning tail:  -35.83
per seed: {'sarsa': [-44.2, -27.3, -25.8], 'q_learning': [-34.3, -37.4, -35.8]}


## 4. The routes differ, not just the numbers

Both greedy paths reach the goal. One hugs the pit; the other keeps a row of margin.

In [5]:
cliff = experiments.make_cliff()
for name in ("sarsa_Q", "q_learning_Q"):
    path = tabular.greedy_path_from_q(cliff, result["last"][name], start=36)
    print(f"{name}: length {len(path)}, first steps {path[:8]}")

sarsa_Q: length 18, first steps [36, 24, 12, 0, 1, 2, 3, 4]
q_learning_Q: length 14, first steps [36, 24, 25, 26, 27, 28, 29, 30]


## 5. Learning curves

Plot the smoothed returns. The gap opens early and persists; averaging over three seeds is what makes it
readable rather than anecdotal.

In [6]:
fig, ax = plt.subplots(figsize=(7, 3.2))
for name in ("sarsa", "q_learning"):
    curves = np.mean([tabular.moving_average(c, 20) for c in result["curves"][name]], axis=0)
    ax.plot(curves, label=name)
ax.set_xlabel("episode"); ax.set_ylabel("smoothed return"); ax.legend(); ax.set_title("cliff walking")
plt.tight_layout(); plt.show()
print("final smoothed values:", {k: round(float(np.mean([tabular.moving_average(c, 20)[-1] for c in result["curves"][k]])), 2) for k in ("sarsa", "q_learning")})

final smoothed values: {'sarsa': -27.22, 'q_learning': -41.42}


C:\Users\Nadeem\AppData\Local\Temp\ipykernel_7660\520285934.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Exercise

1. Add `tabular.expected_sarsa` and `tabular.double_q_learning` to the comparison. Does either close the gap?
2. Sweep $\alpha \in \{0.05, 0.5, 0.9\}$ on cliff walking and report where learning becomes unstable.
3. Explain in three sentences why a lower $\varepsilon$ narrows the gap.

## Self-assessment

- Which algorithm would you deploy on a robot that keeps exploring at runtime?
- What does the tail mean measure that the best episode does not?
- Why is a single seed insufficient for this comparison?